# The Babel Effect on Kaggle
One cell does everything (`kaggle/driver.py`, in its own process): install, checks, model download, number-format check,
then the paraphrase check, the new-facts run and the shared-edit test spread over every GPU, then merging, analyses and packaging.
Rerunning with this notebook's earlier output attached resumes where it stopped. Results: `babel_new_results.zip` in the output.

In [ ]:
import glob, os, subprocess, sys
HOURS = 10.5  # time budget; jobs stop cleanly after this so the session can save and package
drv = sorted(glob.glob("/kaggle/input/**/kaggle/driver.py", recursive=True))
if not drv:
    print("The babel-bundle dataset (with kaggle/driver.py) is not attached to this notebook.")
else:
    env = dict(os.environ, HOURS=str(HOURS), PYTHONUNBUFFERED="1")
    try:
        from kaggle_secrets import UserSecretsClient
        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN secret not readable here (" + type(e).__name__ + "); the driver will try again itself.")
    p = subprocess.Popen([sys.executable, drv[0]], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("driver exit code:", p.returncode)

In [ ]:
import json, os
f = "/kaggle/working/results/RUN_SUMMARY.json"
print(json.dumps(json.load(open(f)), indent=1) if os.path.exists(f) else "no summary yet (see the messages above)")